# Residual Disagreement Analysis Between Subscription and Embedding Graphs

This notebook extends `graph_similarity_analysis.ipynb` by explicitly modeling disagreement between the two directed, row-normalized distance spaces:

- `emb_analysis` (semantic distance)
- `sub_analysis` (audience-overlap-derived distance)

We define a **directed residual matrix**:

\[
R = sub\_analysis - emb\_analysis
\]

A positive residual means the subscriptions graph considers a destination pair **farther** than semantics suggests (because these are distances). A negative residual means subscriptions considers the pair **closer** than semantics suggests.

The notebook operates at three levels:

1. **Pair level**: rank ordered pairs by absolute residual \(|R_{ij}|\), plus row-standardized residuals.
2. **Node level**: compute outgoing and incoming divergence aggregates.
3. **Meso level**: cluster residual fingerprints (rows of \(R\)) and build a signed directed residual edge list.


## 1) Setup and imports

The code below is intentionally verbose and heavily commented to make each transformation auditable.


In [ ]:
from __future__ import annotations

import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)


## 2) Configuration

You can point this notebook to:

- a specific similarity-analysis run version (`SIMILARITY_RUN_VERSION`), or
- the `latest` similarity run.

By default we read from:
`/content/drive/MyDrive/Graphiko/analysis/graph_similarity/latest/`


In [ ]:
# Root folder containing outputs produced by graph_similarity_analysis.ipynb.
SIMILARITY_ANALYSIS_ROOT = Path('/content/drive/MyDrive/Graphiko/analysis/graph_similarity')

# If set to None, the notebook reads from <SIMILARITY_ANALYSIS_ROOT>/latest.
# If set to a string like 'v20260422_120000', it reads that exact run folder.
SIMILARITY_RUN_VERSION = None

# Root folder where THIS notebook writes its own residual-analysis outputs.
RESIDUAL_ANALYSIS_ROOT = Path('/content/drive/MyDrive/Graphiko/analysis/residual_disagreement')
RESIDUAL_RUN_VERSION = None  # None => auto timestamp in UTC

# Residual orientation (must remain consistent throughout all downstream metrics).
# Chosen orientation: subscriptions minus embeddings.
RESIDUAL_FORMULA = 'sub_minus_emb'

# Signed-graph extraction: keep edges whose |R_ij| is in the top X%.
TOP_ABS_RESIDUAL_PERCENTILE = 95

# Candidate k values for clustering residual fingerprints.
CLUSTER_K_CANDIDATES = [2, 3, 4, 5, 6, 7, 8]
RANDOM_STATE = 42


## 3) Utility functions


In [ ]:
def resolve_similarity_run_dir(root: Path, run_version: str | None) -> Path:
    """Resolve a similarity-analysis run directory from run version or latest."""
    if run_version:
        run_dir = root / run_version
    else:
        run_dir = root / 'latest'
    if not run_dir.exists():
        raise FileNotFoundError(f'Similarity run directory not found: {run_dir}')
    return run_dir


def resolve_residual_output_dirs(root: Path, run_version: str | None):
    """Create output directories for this run + a stable latest mirror."""
    if run_version is None:
        ts = datetime.now(timezone.utc).strftime('v%Y%m%d_%H%M%S')
    else:
        ts = run_version

    run_dir = root / ts
    latest_dir = root / 'latest'
    run_dir.mkdir(parents=True, exist_ok=True)
    latest_dir.mkdir(parents=True, exist_ok=True)
    return run_dir, latest_dir, ts


def load_square_matrix(csv_path: Path) -> pd.DataFrame:
    """
    Load a labeled square adjacency matrix from graphiko.adjacency format.

    Expected format:
    - first column = row node id (often 'row_node_id')
    - remaining columns = target node ids
    """
    df = pd.read_csv(csv_path)
    row_id_col = df.columns[0]
    df = df.set_index(row_id_col)

    # Ensure all numeric entries; coerce invalid values to NaN then fill with zero.
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0.0)

    # Enforce row/column order equality when possible (intersection in stable order).
    common = [c for c in df.columns if c in df.index]
    if len(common) < 3:
        raise ValueError('Matrix has fewer than 3 overlapping row/column node ids.')
    df = df.loc[common, common]

    return df


def align_matrices(a: pd.DataFrame, b: pd.DataFrame):
    """Return two matrices aligned to the sorted node intersection."""
    common = sorted(set(a.index).intersection(set(b.index)))
    if len(common) < 3:
        raise ValueError('Need at least 3 shared nodes for residual analysis.')
    return a.loc[common, common], b.loc[common, common]


def sync_latest_outputs(run_dir: Path, latest_dir: Path):
    """Replace latest folder contents with current run artifacts."""
    for child in latest_dir.iterdir():
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()
    for child in run_dir.iterdir():
        dst = latest_dir / child.name
        if child.is_dir():
            shutil.copytree(child, dst)
        else:
            shutil.copy2(child, dst)


## 4) Load matrices from similarity-analysis artifacts


In [ ]:
similarity_run_dir = resolve_similarity_run_dir(SIMILARITY_ANALYSIS_ROOT, SIMILARITY_RUN_VERSION)
output_dir, latest_output_dir, effective_run_version = resolve_residual_output_dirs(
    RESIDUAL_ANALYSIS_ROOT,
    RESIDUAL_RUN_VERSION,
)

summary_path = similarity_run_dir / 'run_summary.json'
if not summary_path.exists():
    raise FileNotFoundError(f'Expected run_summary.json at: {summary_path}')

with summary_path.open('r', encoding='utf-8') as f:
    similarity_summary = json.load(f)

# Prefer exactly the downloaded inputs recorded by graph_similarity_analysis.
emb_csv = Path(similarity_summary['inputs']['embeddings_downloaded_csv'])
sub_csv = Path(similarity_summary['inputs']['subscriptions_downloaded_csv'])

# Fallback to graph roots if needed.
if not emb_csv.exists():
    emb_csv = Path('/content/drive/MyDrive/Graphiko/graphs/embeddings_distance/latest/adjacency_matrix.csv')
if not sub_csv.exists():
    sub_csv = Path('/content/drive/MyDrive/Graphiko/graphs/subscriptions_normalized_distance/latest/adjacency_matrix.csv')

emb_raw = load_square_matrix(emb_csv)
sub_raw = load_square_matrix(sub_csv)
emb_analysis, sub_analysis = align_matrices(emb_raw, sub_raw)

print(f'Similarity input run dir: {similarity_run_dir}')
print(f'Embeddings matrix path:  {emb_csv}')
print(f'Subscriptions matrix path:{sub_csv}')
print(f'Aligned node count: {emb_analysis.shape[0]}')


## 5) Build residual matrix and row-standardized residuals


In [ ]:
if RESIDUAL_FORMULA != 'sub_minus_emb':
    raise ValueError("This notebook is configured for RESIDUAL_FORMULA='sub_minus_emb'.")

# Directed residual matrix (same node ordering for rows/columns).
R = sub_analysis - emb_analysis

# Row-wise mean and std for standardization.
row_means = R.mean(axis=1)
row_stds = R.std(axis=1, ddof=0)

# Avoid divide-by-zero when a row has constant residuals.
safe_row_stds = row_stds.replace(0.0, np.nan)
Z = R.sub(row_means, axis=0).div(safe_row_stds, axis=0).fillna(0.0)

print('Residual matrix shape:', R.shape)
print('Residual matrix summary:')
print(R.stack().describe())


## 6) Pair-level analysis: rank ordered pairs by absolute residual


In [ ]:
# Keep all off-diagonal ordered pairs (i -> j, where i != j).
pair_records = []
for src in R.index:
    for dst in R.columns:
        if src == dst:
            continue
        residual = float(R.loc[src, dst])
        z_value = float(Z.loc[src, dst])
        pair_records.append(
            {
                'source_node_id': src,
                'target_node_id': dst,
                'residual_sub_minus_emb': residual,
                'abs_residual': abs(residual),
                'row_standardized_residual_z': z_value,
            }
        )

pairs_df = pd.DataFrame(pair_records).sort_values('abs_residual', ascending=False).reset_index(drop=True)

pairs_path = output_dir / 'pair_level_residual_ranking.csv'
pairs_df.to_csv(pairs_path, index=False)

print('Top 20 ordered pairs by |residual|:')
display(pairs_df.head(20))
print(f'Saved pair ranking: {pairs_path}')


## 7) Node-level analysis: outgoing and incoming divergence


In [ ]:
# out_divergence_i = sum_j |R_ij|
out_divergence = R.abs().sum(axis=1)

# in_divergence_j = sum_i |R_ij|  (sum over incoming to node j)
in_divergence = R.abs().sum(axis=0)

nodes_df = pd.DataFrame(
    {
        'node_id': R.index,
        'out_divergence': out_divergence.values,
        'in_divergence': in_divergence.reindex(R.index).values,
    }
)

# Additional useful diagnostics for interpretation.
nodes_df['net_divergence'] = nodes_df['out_divergence'] - nodes_df['in_divergence']
nodes_df['total_divergence'] = nodes_df['out_divergence'] + nodes_df['in_divergence']

nodes_sorted_path = output_dir / 'node_level_divergence.csv'
nodes_df.sort_values('total_divergence', ascending=False).to_csv(nodes_sorted_path, index=False)

print('Top 20 nodes by total divergence:')
display(nodes_df.sort_values('total_divergence', ascending=False).head(20))
print(f'Saved node divergence table: {nodes_sorted_path}')


## 8) Meso-level analysis: cluster residual fingerprints (rows of R)


In [ ]:
# Each row of R is a channel's directed divergence fingerprint to all destinations.
X = R.values

# Standardize features (destination columns) to avoid columns with larger variance dominating k-means.
X_scaled = StandardScaler(with_mean=True, with_std=True).fit_transform(X)

# Evaluate candidate k using silhouette score and pick the best.
valid_scores = []
for k in CLUSTER_K_CANDIDATES:
    if k >= len(R):
        continue
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    valid_scores.append({'k': k, 'silhouette': float(score)})

if not valid_scores:
    raise ValueError('No valid clustering candidate k values. Increase node count or lower k candidates.')

scores_df = pd.DataFrame(valid_scores).sort_values('silhouette', ascending=False).reset_index(drop=True)
best_k = int(scores_df.iloc[0]['k'])

best_model = KMeans(n_clusters=best_k, n_init=30, random_state=RANDOM_STATE)
cluster_labels = best_model.fit_predict(X_scaled)

clusters_df = pd.DataFrame({'node_id': R.index, 'residual_cluster': cluster_labels})
cluster_sizes_df = clusters_df['residual_cluster'].value_counts().rename_axis('residual_cluster').reset_index(name='size')

scores_path = output_dir / 'cluster_model_selection.csv'
clusters_path = output_dir / 'residual_fingerprint_clusters.csv'
cluster_sizes_path = output_dir / 'residual_cluster_sizes.csv'

scores_df.to_csv(scores_path, index=False)
clusters_df.to_csv(clusters_path, index=False)
cluster_sizes_df.to_csv(cluster_sizes_path, index=False)

print('Cluster model selection (top rows):')
display(scores_df.head(10))
print(f'Chosen k: {best_k}')
print('Cluster sizes:')
display(cluster_sizes_df.sort_values('size', ascending=False))


## 9) Signed directed residual graph (thresholded top residuals)


In [ ]:
threshold_value = np.percentile(pairs_df['abs_residual'].values, TOP_ABS_RESIDUAL_PERCENTILE)

signed_edges = pairs_df[pairs_df['abs_residual'] >= threshold_value].copy()
signed_edges['edge_sign'] = np.where(
    signed_edges['residual_sub_minus_emb'] > 0,
    'positive_sub_farther_than_emb',
    'negative_sub_closer_than_emb',
)

signed_edges_path = output_dir / 'signed_residual_edges_top_percentile.csv'
signed_edges.to_csv(signed_edges_path, index=False)

print(f'Absolute residual threshold (p{TOP_ABS_RESIDUAL_PERCENTILE}): {threshold_value:.8f}')
print(f'Number of signed edges retained: {len(signed_edges)} / {len(pairs_df)}')
display(signed_edges.head(20))


## 10) Visual diagnostics


In [ ]:
# Heatmap of raw residual matrix.
fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
sns.heatmap(R, cmap='coolwarm', center=0.0, ax=ax)
ax.set_title('Directed residual matrix R = subscriptions - embeddings')
residual_heatmap_path = output_dir / 'residual_matrix_heatmap.png'
fig.savefig(residual_heatmap_path, dpi=220)
plt.show()

# Heatmap of row-standardized residuals.
fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
sns.heatmap(Z, cmap='vlag', center=0.0, ax=ax)
ax.set_title('Row-standardized residual matrix Z')
residual_z_heatmap_path = output_dir / 'residual_z_heatmap.png'
fig.savefig(residual_z_heatmap_path, dpi=220)
plt.show()

# Top nodes by divergence.
top_nodes = nodes_df.sort_values('total_divergence', ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
sns.barplot(data=top_nodes, x='node_id', y='total_divergence', color='#4c72b0', ax=ax)
ax.set_title('Top 20 nodes by total residual divergence')
ax.tick_params(axis='x', rotation=75)
node_div_plot_path = output_dir / 'node_total_divergence_top20.png'
fig.savefig(node_div_plot_path, dpi=220)
plt.show()

print('Saved plots:')
print('-', residual_heatmap_path)
print('-', residual_z_heatmap_path)
print('-', node_div_plot_path)


## 11) Persist run summary and sync `latest`


In [ ]:
run_summary = {
    'residual_run_version': effective_run_version,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'residual_formula': RESIDUAL_FORMULA,
    'inputs': {
        'similarity_run_dir': str(similarity_run_dir),
        'similarity_summary_json': str(summary_path),
        'embeddings_matrix_csv': str(emb_csv),
        'subscriptions_matrix_csv': str(sub_csv),
        'aligned_node_count': int(R.shape[0]),
    },
    'parameters': {
        'top_abs_residual_percentile': float(TOP_ABS_RESIDUAL_PERCENTILE),
        'cluster_k_candidates': [int(x) for x in CLUSTER_K_CANDIDATES],
        'random_state': int(RANDOM_STATE),
    },
    'outputs': {
        'pair_level_ranking_csv': str(pairs_path),
        'node_level_divergence_csv': str(nodes_sorted_path),
        'cluster_model_selection_csv': str(scores_path),
        'residual_fingerprint_clusters_csv': str(clusters_path),
        'residual_cluster_sizes_csv': str(cluster_sizes_path),
        'signed_residual_edges_csv': str(signed_edges_path),
        'residual_heatmap_png': str(residual_heatmap_path),
        'residual_z_heatmap_png': str(residual_z_heatmap_path),
        'node_total_divergence_top20_png': str(node_div_plot_path),
    },
}

summary_out_path = output_dir / 'run_summary.json'
with summary_out_path.open('w', encoding='utf-8') as f:
    json.dump(run_summary, f, indent=2)

sync_latest_outputs(output_dir, latest_output_dir)

print('Residual disagreement analysis complete.')
print(json.dumps(run_summary, indent=2))
